**UTS Data Mining II · Text**

Pilih satu input: CSV, folder TXT, ZIP TXT, atau NPZ. Hapus input lain sebelum Run All, lalu lanjut ke **Cek data**. Ganti path dan kolom `content` sesuai soal.

In [ ]:
# Jalankan sekali jika diperlukan:
# %pip install numpy pandas matplotlib scikit-learn nltk openpyxl
# import nltk
# nltk.download("stopwords")

**Baca data CSV**

In [ ]:
import pandas as pd

df = pd.read_csv("/content/BSI-reviews.csv")
display(df.head())

**Baca folder TXT**

Satu TXT menjadi satu dokumen, termasuk subfolder. Ganti encoding bila bukan UTF-8.

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("/content/data/text")
files = sorted([p for p in data_dir.rglob("*") if p.suffix.lower() == ".txt"])
texts = [file.read_text(encoding="utf-8-sig") for file in files]

df = pd.DataFrame({"path": [str(p) for p in files], "content": texts})
print("Jumlah dokumen:", len(df))
display(df.head())

**Baca ZIP TXT**

Ekstrak ke folder baru agar data ZIP berbeda tidak bercampur; subfolder ikut dibaca.

In [ ]:
import zipfile
from pathlib import Path
import pandas as pd

zip_path = "/content/text.zip"
data_dir = Path("/content/text_extracted")
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(data_dir)

files = sorted([p for p in data_dir.rglob("*") if p.suffix.lower() == ".txt"])
texts = [file.read_text(encoding="utf-8-sig") for file in files]

df = pd.DataFrame({"path": [str(p) for p in files], "content": texts})
print("Jumlah dokumen:", len(df))
display(df.head())

**Baca NPZ teks**

Gunakan array teks Unicode satu dimensi. Cek `data.files`, ganti key `texts` sesuai file, lalu lanjut ke **Cek data**.

In [ ]:
import numpy as np
import pandas as pd

data = np.load("/content/text.npz", allow_pickle=False)
print("Nama array:", data.files)

texts = data["texts"]
df = pd.DataFrame({"content": texts})
print("Jumlah dokumen:", len(df))
display(df.head())

**Cek data**

In [ ]:
df.info()
display(df.isnull().sum())

df = df.dropna(subset=["content"]).copy()
df["content"] = df["content"].astype(str)

**Cleaning**

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return " ".join(text.split())

df["cleaned_content"] = df["content"].apply(clean_text)
display(df[["content", "cleaned_content"]].head())

**Tokenisasi**

Pisahkan teks bersih berdasarkan spasi.

In [ ]:
df["tokens"] = df["cleaned_content"].str.split()
display(df[["cleaned_content", "tokens"]].head())

**Stopword NLTK**

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("indonesian"))
df["tokens_no_stopwords"] = df["tokens"].apply(lambda tokens: [word for word in tokens if word not in stop_words])
df["text_processed"] = df["tokens_no_stopwords"].apply(" ".join)
df = df[df["text_processed"].ne("")].reset_index(drop=True)
display(df[["content", "text_processed"]].head())

**TF-IDF**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
X = vectorizer.fit_transform(df["text_processed"])
print("Shape TF-IDF:", X.shape)

**K-Means**

Ganti `k` sesuai soal; minimal 2 dan lebih kecil dari jumlah dokumen.

In [ ]:
from sklearn.cluster import KMeans

k = 7
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X)
display(df[["content", "cluster"]].head())

**Evaluasi**

Silhouette lebih tinggi berarti pemisahan lebih baik. Maksimal 3.000 dokumen dipakai untuk perkiraan skor.

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X, df["cluster"], sample_size=min(3000, len(df)), random_state=42)
print("Silhouette score:", score)

**Grafik cluster**

SVD hanya untuk grafik; clustering tetap memakai TF-IDF.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=42)
X_plot = svd.fit_transform(X)

plt.figure(figsize=(8, 6))
points = plt.scatter(X_plot[:, 0], X_plot[:, 1], c=df["cluster"], cmap="viridis", alpha=0.6)
plt.legend(*points.legend_elements(), title="Cluster")
plt.xlabel("Komponen SVD 1")
plt.ylabel("Komponen SVD 2")
plt.title("Cluster teks")
plt.show()

**Kata utama dan contoh teks tiap cluster**

In [ ]:
terms = vectorizer.get_feature_names_out()
display(df["cluster"].value_counts().sort_index())

for cluster in range(k):
    top_indices = kmeans.cluster_centers_[cluster].argsort()[-10:][::-1]
    print(f"Cluster {cluster}:", ", ".join(terms[top_indices]))
    display(df.loc[df["cluster"] == cluster, ["content"]].head(3))

**Simpan hasil (opsional)**

Hapus `#` pada kode untuk menyimpan CSV.

In [ ]:
# df.to_csv("text_hasil.csv", index=False)

**Alternatif Excel dan Kaggle**
Ganti loader CSV dengan salah satu contoh ini; kolom teks tetap `content`.

**Excel**
```python
import pandas as pd
df = pd.read_excel("/content/text.xlsx")
display(df.head())
```

**Kaggle CSV** — install `kagglehub`; ganti slug dan nama file.
```python
import pandas as pd
import kagglehub
from pathlib import Path

folder = Path(kagglehub.dataset_download("pemilik/nama-dataset"))
print(list(folder.iterdir()))
df = pd.read_csv(folder / "data.csv")
display(df.head())
```
Untuk Excel, gunakan `pd.read_excel(...)`.